# Plaka Demo - YOLO + EasyOCR

Bu notebook, eğitilmiş `best.pt` modeli ve `ornek_arac.jpg` görseliyle uçtan uca demo çıktısı üretir. Kaggle'da Input kısmına şu dosyalar eklenmelidir:

- `plaka_projesi_ciktilar.zip` veya doğrudan `best.pt`
- `ornek_arac.jpg`

## 1. Paket Kurulumu

In [ ]:
!pip install -q ultralytics easyocr opencv-python-headless pandas matplotlib

## 2. Dosyaları Bulma

Notebook, `/kaggle/input` ve `/kaggle/working` altında `best.pt`, `plaka_projesi_ciktilar.zip` ve `ornek_arac.jpg` dosyalarını otomatik arar.

In [ ]:
from pathlib import Path
import zipfile

WORK_DIR = Path('/kaggle/working')
OUTPUT_DIR = WORK_DIR / 'plaka_demo_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_first(patterns):
    for base in [Path('/kaggle/input'), Path('/kaggle/working')]:
        for pattern in patterns:
            matches = sorted(base.rglob(pattern))
            if matches:
                return matches[0]
    return None

zip_path = find_first(['plaka_projesi_ciktilar.zip'])
if zip_path:
    extract_dir = WORK_DIR / 'trained_model_zip'
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_file:
        zip_file.extractall(extract_dir)
    print('Zip çıkarıldı:', zip_path)

weights_path = find_first(['best.pt'])
image_path = find_first(['ornek_arac.jpg', 'ornek_arac.jpeg', 'ornek_arac.png'])

print('Model:', weights_path)
print('Görsel:', image_path)

if weights_path is None:
    raise FileNotFoundError('best.pt bulunamadı. Input kısmına best.pt veya plaka_projesi_ciktilar.zip ekleyin.')
if image_path is None:
    raise FileNotFoundError('ornek_arac.jpg bulunamadı. Input kısmına örnek araç fotoğrafını ekleyin.')

## 3. YOLO ile Plaka Tespiti ve EasyOCR ile Okuma

In [ ]:
from ultralytics import YOLO
import easyocr
import cv2
import pandas as pd
import re
from IPython.display import display, Image

def normalize_plate_text(text):
    text = text.upper()
    replacements = {'İ': 'I', 'Ş': 'S', 'Ğ': 'G', 'Ü': 'U', 'Ö': 'O', 'Ç': 'C'}
    for source, target in replacements.items():
        text = text.replace(source, target)
    return re.sub(r'[^A-Z0-9]', '', text)

def preprocess_for_ocr(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    denoised = cv2.bilateralFilter(enhanced, d=7, sigmaColor=50, sigmaSpace=50)
    threshold = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 5)
    return [crop, gray, enhanced, threshold]

model = YOLO(str(weights_path))
image = cv2.imread(str(image_path))
if image is None:
    raise ValueError(f'Görsel okunamadı: {image_path}')

pred = model.predict(source=image, conf=0.25, imgsz=640, verbose=False)[0]
reader = easyocr.Reader(['en'], gpu=True)

rows = []
annotated = image.copy()

if pred.boxes is None or len(pred.boxes) == 0:
    status = 'tespit_edilemedi'
    cv2.putText(annotated, 'Plaka tespit edilemedi', (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
    rows.append({'image_path': str(image_path), 'detected_text': '', 'ocr_confidence': 0.0, 'detection_confidence': 0.0, 'status': status})
else:
    boxes = pred.boxes
    best_index = int(boxes.conf.argmax().item())
    box = boxes.xyxy[best_index].cpu().numpy().astype(int)
    detection_conf = float(boxes.conf[best_index].cpu().item())
    x1, y1, x2, y2 = box.tolist()
    h, w = image.shape[:2]
    pad_x = int((x2 - x1) * 0.06)
    pad_y = int((y2 - y1) * 0.06)
    x1p, y1p = max(0, x1 - pad_x), max(0, y1 - pad_y)
    x2p, y2p = min(w, x2 + pad_x), min(h, y2 + pad_y)
    crop = image[y1p:y2p, x1p:x2p].copy()
    crop_path = OUTPUT_DIR / 'ornek_arac_plate_crop.jpg'
    cv2.imwrite(str(crop_path), crop)

    candidates = []
    for variant in preprocess_for_ocr(crop):
        for item in reader.readtext(variant, detail=1, paragraph=False, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'):
            raw_text = str(item[1])
            confidence = float(item[2])
            cleaned = normalize_plate_text(raw_text)
            if cleaned:
                candidates.append((cleaned, confidence, raw_text))

    if candidates:
        detected_text, ocr_conf, raw_text = sorted(candidates, key=lambda item: (item[1], len(item[0])), reverse=True)[0]
        status = 'basarili' if ocr_conf >= 0.50 else 'dusuk_guven'
    else:
        detected_text, ocr_conf, raw_text = '', 0.0, ''
        status = 'ocr_okunamadi'

    cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 180, 0), 2)
    cv2.putText(annotated, f'{detected_text or "OCR okunamadi"} det:{detection_conf:.2f}', (x1, max(20, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 180, 0), 2)
    rows.append({
        'image_path': str(image_path),
        'detected_text': detected_text,
        'raw_ocr_text': raw_text,
        'ocr_confidence': round(ocr_conf, 4),
        'detection_confidence': round(detection_conf, 4),
        'status': status,
        'crop_path': str(crop_path),
    })

output_path = OUTPUT_DIR / 'demo_result.jpg'
csv_path = OUTPUT_DIR / 'results.csv'
cv2.imwrite(str(output_path), annotated)
pd.DataFrame(rows).to_csv(csv_path, index=False)

print(pd.DataFrame(rows))
display(Image(filename=str(output_path)))
print('Çıktı klasörü:', OUTPUT_DIR)

## 4. Demo Çıktısını Zip Yapma

In [ ]:
!cd /kaggle/working && zip -r plaka_demo_outputs.zip plaka_demo_outputs
!ls -lh /kaggle/working/plaka_demo_outputs.zip